# QGNN Batched — Aer / IBM Quantum

Uses **`QGNNGraphClassifier_Batched`** (vectorized: 1 QNode call per hop instead of N_nodes × hop).

Device options:
- `qiskit.aer` — local Aer simulator (fast, no account needed)
- `qiskit.remote` — real IBM backend via `qiskit_ibm_runtime`

Set `USE_REMOTE = False` to run locally with Aer.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

from data import load_dataset
from utils import train_graph, test_graph
from model import QGNNGraphClassifier_Batched

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PennyLane : {qml.__version__}")
print(f"PyTorch   : {torch.__version__}")
print(f"Torch dev : {DEVICE}")

PennyLane : 0.38.0
PyTorch   : 2.7.0+cu126
Torch dev : cuda


In [2]:
# ── Qubit / PQC config ──────────────────────────────────────────────────────
GRAPHLET_SIZE  = 4     # nodes per subgraph  =  node qubits
NUM_ENT_LAYERS = 1     # entangling layers per PQC
HOP_NEIGHBOR   = 2     # GNN message-passing hops

node_qubit = GRAPHLET_SIZE
edge_qubit = node_qubit - 1
n_qubits   = node_qubit + edge_qubit      # data qubits (7 for graphlet_size=4)
n_wires    = n_qubits + 2                 # +2 ancilla

w_shapes_dict = {
    'spreadlayer': (0, n_qubits, 1),
    'inits'      : (NUM_ENT_LAYERS, 4),
    'strong'     : (NUM_ENT_LAYERS, 4),
    'update'     : (GRAPHLET_SIZE, max(NUM_ENT_LAYERS - 1, 1), 4, 3),
    'twodesign'  : (0, NUM_ENT_LAYERS, 1, 2),
}

print(f"Wires : {n_wires}  ({node_qubit} node + {edge_qubit} edge + 2 ancilla)")

Wires : 9  (4 node + 3 edge + 2 ancilla)


In [3]:
# ── Dataset ──────────────────────────────────────────────────────────────────
DATASET    = 'MUTAG'
TRAIN_SIZE = 100
TEST_SIZE  = 50
BATCH_SIZE = 32

dataset, train_loader, test_loader, task_type = load_dataset(
    name=DATASET,
    path='../data',
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    eval_size=0,
    batch_size=BATCH_SIZE,
)

node_input_dim = dataset[0].x.shape[1]        if dataset[0].x         is not None else 1
edge_input_dim = dataset[0].edge_attr.shape[1] if dataset[0].edge_attr is not None else 1
num_classes    = dataset.num_classes

print(f"Dataset      : {DATASET}  ({num_classes} classes)")
print(f"Node feat dim: {node_input_dim}")
print(f"Edge feat dim: {edge_input_dim}")
print(f"Train / Test : {len(train_loader.dataset)} / {len(test_loader.dataset)}")

Dataset      : MUTAG  (2 classes)
Node feat dim: 7
Edge feat dim: 4
Train / Test : 100 / 50


In [4]:
# ── Device selection ─────────────────────────────────────────────────────────
# Requires: pip install pennylane-qiskit
USE_REMOTE = False   # True → real IBM backend via qiskit_ibm_runtime
SHOTS      = 1024

if USE_REMOTE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService()
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=n_wires)
    print(f"Remote backend: {backend.name}")
    q_dev = qml.device('qiskit.remote', wires=n_wires, shots=SHOTS, backend=backend)
else:
    q_dev = qml.device('qiskit.aer', wires=n_wires, shots=SHOTS)
    print(f"Aer local  | wires={n_wires} | shots={SHOTS}")

Aer local  | wires=9 | shots=1024


In [5]:
# ── Model ─────────────────────────────────────────────────────────────────────
# QGNNGraphClassifier_Batched stacks all subgraph inputs → single batched
# QNode call per hop (via qml.batch_input), then runs upd MLP on full batch.
HIDDEN_DIM = 64

model = QGNNGraphClassifier_Batched(
    q_dev          = q_dev,
    w_shapes       = w_shapes_dict,
    hidden_dim     = HIDDEN_DIM,
    node_input_dim = node_input_dim,
    edge_input_dim = edge_input_dim,
    graphlet_size  = GRAPHLET_SIZE,
    hop_neighbor   = HOP_NEIGHBOR,
    num_classes    = num_classes,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params}")

Hidden dim: 64
Trainable parameters: 7234


In [6]:
# ── (Optional) Load pre-trained checkpoint ────────────────────────────────────
CHECKPOINT = None   # e.g. '../results/model/mutag_model_<timestamp>.pt'

if CHECKPOINT is not None:
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded: {CHECKPOINT}")
else:
    print("No checkpoint — training from scratch")

No checkpoint — training from scratch


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS    = 20
LR        = 5e-2
STEP_SIZE = 5
GAMMA     = 0.8

optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)
criterion = nn.CrossEntropyLoss()

train_losses, test_losses = [], []
train_accs,   test_accs   = [], []

print(f"{'Ep':>3}  {'TrLoss':>8}  {'TrAcc':>7}  {'TeLoss':>8}  {'TeAcc':>7}  {'s/ep':>6}")
print("-" * 48)

with qml.Tracker(q_dev) as tracker:
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_graph(model, optimizer, train_loader, criterion, DEVICE)
        tr_loss, tr_acc, _ = test_graph(model, train_loader, criterion, DEVICE, num_classes)
        te_loss, te_acc, _ = test_graph(model, test_loader,  criterion, DEVICE, num_classes)
        scheduler.step()

        train_losses.append(tr_loss);  test_losses.append(te_loss)
        train_accs.append(tr_acc);     test_accs.append(te_acc)

        print(f"{epoch:3d}  {tr_loss:8.4f}  {tr_acc:7.4f}  {te_loss:8.4f}  {te_acc:7.4f}  {time.time()-t0:6.1f}")

print("\nQNode execution totals:", tracker.totals)

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
ep = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle(f"QGNNGraphClassifier_Batched  |  {DATASET}  |  Aer shots={SHOTS}", fontsize=13)

axes[0].plot(ep, train_losses, label='Train')
axes[0].plot(ep, test_losses,  label='Test')
axes[0].set_title('Loss');  axes[0].set_xlabel('Epoch');  axes[0].legend()

axes[1].plot(ep, train_accs, label='Train')
axes[1].plot(ep, test_accs,  label='Test')
axes[1].set_title('Accuracy');  axes[1].set_xlabel('Epoch');  axes[1].legend()

plt.tight_layout()
os.makedirs('../results/fig', exist_ok=True)
plt.savefig('../results/fig/aer_batched_mutag.png', dpi=150)
plt.show()

print(f"Final test acc : {test_accs[-1]:.4f}")
print(f"Best  test acc : {max(test_accs):.4f}  (epoch {test_accs.index(max(test_accs)) + 1})")